In [ ]:
# Imports

# Import Python packages
# import sys
import os
# import subprocess
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const#
from pprint import pprint
from copy import deepcopy

# Import the Simulation packages
from spinsimulation import *

from willkelabexchange import Tools, Plotting, PowerPointControl, FileManagement, Fitting
from datetime import datetime


# Experiment

In [ ]:
from scipy.io import loadmat


from willkelabexchange import Tools, Plotting, FileManagement
plotting_cfg_path = r"\\sccfs.scc.kit.edu\oe\PHI\GroupWillke\Papers\MoS2_Defects\Evaluation\plotting_config.pkl"
wle_plotting = Plotting(plotting_config_path=plotting_cfg_path, save_output= False)


# ## read plotting_config
plotting_cfg = Tools.load_pkl(folder=r"\\sccfs.scc.kit.edu\oe\PHI\GroupWillke\Papers\MoS2_Defects\Evaluation", filename="plotting_config.pkl")
#pprint(plotting_cfg["folders"])


# === Define the folder path ===
path = r"\\sccfs.scc.kit.edu\oE\PHI\GroupWillke\Papers\MoS2_Defects\Evaluation\Supplement\Additional data on the C_s defect (Wantong)\Bfield_dIdV"

file_list = [["RANGE","BiasSpectr_20250826_154612", "BiasSpectr_20250826_154935"],#"_1000mT"
            ["RANGE","BiasSpectr_20250826_174522", "BiasSpectr_20250826_174903"] #"_0mT"
]

plot_return = wle_plotting.plot_parameter_sweep(files=file_list, parameter="B-Field",
                                  plotstyle="2dLines",
                                  subtract_background="slope",
                                  exclude_values_background=[-0.5, 0.5],
                                  offset= 000e-15
                                  )

# Extract x and y data
x=plot_return['data'].index
y=plot_return['data'].values[:,1]/plot_return['data'].values[0,1]

# Plot the data
plt.figure(figsize=(8,5))
plt.plot(x, y, '-', lw=1, color='red', label='Complex Center')
#plt.plot(x2, y2, '-', lw=1, color='blue', label='Complex Fe')
plt.legend()
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()



# Simulation

In [ ]:
# Define the Spin System
spinsys=SpinSys([0.5])

# Basic Parameters
spinsys.B=[0,0,1]
spinsys.BTip=[1e-9,0,0]
spinsys.tip='f'
spinsys.T=0.2


# Tunnel Parameters
spinsys.V_DC=100
spinsys.TipPolarization = [0,0,0]
spinsys.U=[0]
spinsys.G=1e-10
tsratio=1
spinsys.G_ss=[spinsys.G*tsratio]
spinsys.G_tt=1

# ESR Parameters
spinsys.ReadoutSpin=1
spinsys.RabiRate=1
spinsys.T2=200


vis.plotIETS(spinsys, Vrange=[-1.5,1.5],N=500, norm=True, order='2nd', plotAllSpins=True);

# Fit

In [ ]:
# Define the fitting function 
def fit_fun_2(x_vals, g, T, U): # x and Open variables

    # Modify the spin system according to the Variables
    spinsys.gfactorvector=[g]
    spinsys.U=[U]
    spinsys.T=T

    
    # Calculate the Energies new
    hamil.calcEigEnergies(spinsys)
    hamil.calcEigStates(spinsys)

    # Shift the x-values by a constant offset
    #x_vals_new = x_vals + x_vals[np.argmin(y)]
    x_vals_new = x_vals - 0.05
    N=len(x_vals)
    V, I=dIdV.calcIETS_2nd(spinsys, Vrange=[np.min(x_vals_new),np.max(x_vals_new)],N=N, norm=True);
    return I

from scipy.optimize import curve_fit


# Initial guess:
initial_guess = [2.0,1,0]

# Bounds:
bounds_custom = [(1.5, 3),  # g
                 (0.05,5),    # T
                 (-1, 1),    # U
                 ]



# Perform the fit
popt, pcov = curve_fit(fit_fun_2, x, y, p0=initial_guess, bounds=([b[0] for b in bounds_custom], [b[1] for b in bounds_custom]))


# Print the fit results
pprint(popt)
perr = np.sqrt(np.diag(pcov))
pprint(perr)

## Plot the Fit

In [ ]:
# Calculate fitted curve
x_fit = np.linspace(np.min(x), np.max(x), 1000)
y_fit=fit_fun_2(x_fit, *popt)

# Plot the results
plt.figure(figsize=(8,5))
plt.plot(x, y, 'k.', markersize=4)
plt.plot(x_fit, y_fit, 'r-', lw=1)
label = (
    "fit:\n"
    f"g={popt[0]:.3g} ± {perr[0]:.3g}\n"
    f"T={popt[1]:.3g} ± {perr[1]:.3g}\n"
    f"U={popt[2]:.3g} ± {perr[2]:.3g}"
)
plt.legend(['data', label], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 2nd + Rate

In [ ]:
y=plot_return['data'].values[:,1]/plot_return['data'].values[0,1]

# Define the fitting function 
def fit_fun_2R(x_vals, g, T, U, R): # x and Open variables

    # Modify the spin system according to the Variables
    spinsys.gfactorvector=[g]
    spinsys.U=[U]
    spinsys.T=T
    spinsys.G=1
    spinsys.G_ss=[R*spinsys.G*tsratio]

    
    # Calculate the Energies new
    hamil.calcEigEnergies(spinsys)
    hamil.calcEigStates(spinsys)

    # Calculate the dIdV spectrum according 
    N=len(x_vals)
    x_vals_new=x_vals + 0.05
    _, I, _=dIdV.calcIETS_2ndR(spinsys, Vrange=[np.min(x_vals_new),np.max(x_vals_new)],N=N, norm=True);
    return I

from scipy.optimize import curve_fit


# Initial guess:
initial_guess = [2.5,1,0.0,100]

# Bounds:
bounds_custom = [(1.5, 3),  # g
                 (0.05,5),    # T
                 (-1, 1),    # U
                 (0.01,1000) # R
                 ]


y_test=fit_fun_2R(x, *initial_guess)

plt.figure(figsize=(8,5))
plt.plot(x, y, 'k.', markersize=4)
plt.plot(x, y_test, 'b-', lw=1)
label = (
    "fit:\n"
    f"g={initial_guess[0]:.3g}\n"
    f"T={initial_guess[1]:.3g}\n"
    f"U={initial_guess[2]:.3g}\n"
    f"R={initial_guess[3]:.3g}"
)
plt.legend(['data',label], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Perform the fit
popt, pcov = curve_fit(fit_fun_2R, x, y, p0=initial_guess, bounds=([b[0] for b in bounds_custom], [b[1] for b in bounds_custom]))

# Print the fit results
pprint(popt)
perr = np.sqrt(np.diag(pcov))
pprint(perr)

In [ ]:
# Calculate fitted curve
x_fit = np.linspace(np.min(x), np.max(x), 1000)
y_fit=fit_fun_2R(x, *popt)

# Plot the results
plt.figure(figsize=(8,5))
plt.plot(x, y, 'k.', markersize=4)
plt.plot(x, y_fit, 'r-', lw=1)
label = (
    "fit:\n"
    f"g={popt[0]:.3g} ± {perr[0]:.3g}\n"
    f"T={popt[1]:.3g} ± {perr[1]:.3g}\n"
    f"U={popt[2]:.3g} ± {perr[2]:.3g}\n"
    f"R={popt[3]:.3g} ± {perr[3]:.3g}\n"
)
plt.legend(['data', label], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 3rd

In [ ]:
# Define the fitting function 
def fit_fun_3(x_vals, g, T, U, J): # x and Open variables

    # Modify the spin system according to the Variables
    spinsys.gfactorvector=[g]
    spinsys.U=[U]
    spinsys.T=T
    spinsys.G=1
    spinsys.J=[J]
    #spinsys.lt=lt #lifetime broadening
    #spinsys.w=w # integration bandwidth
    

    
    # Calculate the Energies new
    hamil.calcEigEnergies(spinsys)
    hamil.calcEigStates(spinsys)
    bath.calcPopulations(spinsys, AllowPumping=False)
    bath.calcRates(spinsys)

    # Calculate the dIdV spectrum according 
    N=len(x_vals)
    x_vals_new=x_vals + 0.05
    _, I = dIdV3.calcIETS_3rd(spinsys, Vrange=[np.min(x_vals_new),np.max(x_vals_new)],N=N, norm=True);
    return I

from scipy.optimize import curve_fit


# Initial guess:
initial_guess = [2.6,0.4,0.05,-0.05]
#initial_guess = [2,1,0,-0.1]

# Bounds:
bounds_custom = [(1.5, 3),  # g
                 (0.05,5),    # T
                 (-1, 1),    # U
                 (-1,1) # J
                 ]



# Guess first
y_test=fit_fun_3(x, *initial_guess)

label = (
    "guess:\n"
    f"g={initial_guess[0]:.3g}\n"
    f"T={initial_guess[1]:.3g}\n"
    f"U={initial_guess[2]:.3g}\n"
    f"J={initial_guess[3]:.3g}"
)

plt.figure(figsize=(8,5))
plt.plot(x, y_test, 'b-', markersize=4, label=label)
plt.plot(x, y, 'k.', lw=1)

plt.legend([ label, 'data'], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Perform the Fit


# Perform the fit
popt, pcov = curve_fit(fit_fun_3, x, y, p0=initial_guess, bounds=([b[0] for b in bounds_custom], [b[1] for b in bounds_custom]))

# Print the fit results
pprint(popt)
perr = np.sqrt(np.diag(pcov))
pprint(perr)

In [ ]:
# Calculate fitted curve
x_fit = np.linspace(np.min(x), np.max(x), 100)
y_fit=fit_fun_3(x, *initial_guess)

# Plot the results
plt.figure(figsize=(8,5))
plt.plot(x, y, 'k.', markersize=4)
plt.plot(x, y_fit, 'r-', lw=1)
label = (
    "fit:\n"
    f"g={popt[0]:.3g} ± {perr[0]:.3g}\n"
    f"T={popt[1]:.3g} ± {perr[1]:.3g}\n"
    f"U={popt[2]:.3g} ± {perr[2]:.3g}\n"
    f"J={popt[3]:.3g} ± {perr[3]:.3g}\n"
)
plt.legend(['data', label], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()